# 07: One loss, two learning systems

**For:** developers comfortable with basic PyTorch. Start with lab 03 if gradients are new to you.

**Your mission:** build a classical encoder followed by a quantum layer, then prove that one optimizer updates both.
You will leave with a fit on held-out inputs, a gradient plot, and a compact experiment report.
This toy problem demonstrates joint training, not a quantum advantage.

**Before running:** if the quantum calculation is in the middle of the model, what must happen for the classical encoder to receive a gradient?

## 1. Make training and test inputs

The target is cos(x), so we know the answer independently of our implementation.
Alternate points go into training and testing. The test set stays inside the training range: this checks interpolation, not extrapolation.

In [ ]:
import torch
import flagquantum as fq
import matplotlib.pyplot as plt

all_x = torch.linspace(-2, 2, 41)
x_train, x_test = all_x[::2], all_x[1::2]
y_train, y_test = torch.cos(x_train), torch.cos(x_test)
plt.scatter(x_train, y_train, label="Train", marker="o")
plt.scatter(x_test, y_test, label="Test", marker="x")
plt.xlabel("Input x")
plt.ylabel("Target cos(x)")
plt.legend()
plt.show()


## 2. Write the quantum program

The classical encoder will produce an angle for each input. We add a trainable quantum offset and apply RY.
The circuit's Z expectation is the model's prediction. `bsz` creates one circuit state for each item in the input batch.

In [ ]:
def quantum_program(parameters, encoded):
    circuit = fq.Circuit(1, bsz=len(encoded))
    circuit.ry(0, encoded[:, 0] + parameters["offset"][0])
    return circuit


quantum = fq.Module(
    quantum_program,
    parameters={"offset": (1,)},
    init="uniform",
    seed=42,
    policy=fq.RuntimePolicy(
        execution_options=fq.ExecutionOptions(mode="statevector", device="cpu"),
        observable="z_sum",
        observable_wires=(0,),
    ),
)


## 3. Connect an ordinary PyTorch layer

The encoder maps x to wx+b. The quantum layer maps that angle to a measured expectation.
Together, this particular model computes cos(wx+b+offset). There is a continuous gradient path through both layers.

**Predict:** can b and offset be determined uniquely from these data? Keep that question in mind when looking at learned parameters.

In [ ]:
encoder = torch.nn.Linear(1, 1)
with torch.no_grad():
    encoder.weight.fill_(0.7)
    encoder.bias.fill_(0.2)
model = torch.nn.Sequential(encoder, quantum)
initial_parameters = {name: p.detach().clone() for name, p in model.named_parameters()}
initial_prediction = model(x_test[:, None]).detach().reshape(-1)
print("Prediction shape:", initial_prediction.shape)
print("Trainable parameters:")
for name, p in model.named_parameters():
    print(name, p.detach())


## 4. Inspect one backward pass before training

A decreasing loss alone does not prove that both layers are learning. First inspect their gradients directly.
Parameter names starting with `0.` belong to the encoder; `1.` belongs to the quantum layer.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.04)
optimizer.zero_grad()
prediction = model(x_train[:, None]).reshape(-1)
loss = (prediction - y_train).square().mean()
loss.backward()
for name, p in model.named_parameters():
    assert p.grad is not None and torch.isfinite(p.grad).all()
    print(name, "gradient:", p.grad)
assert encoder.weight.grad.abs().sum().item() > 0
assert sum(p.grad.abs().sum().item() for p in quantum.parameters()) > 0


## 5. Train with one optimizer

Record both losses and the gradient norm of each layer. We evaluate the test inputs without using their errors to update parameters.
Rerun from model construction for a fresh experiment; rerunning only this cell continues training the existing model.

In [ ]:
train_losses, test_losses = [], []
encoder_gradients, quantum_gradients = [], []
for step in range(150):
    optimizer.zero_grad()
    prediction = model(x_train[:, None]).reshape(-1)
    loss = (prediction - y_train).square().mean()
    loss.backward()
    encoder_gradients.append(
        sum(p.grad.square().sum().item() for p in encoder.parameters()) ** 0.5
    )
    quantum_gradients.append(
        sum(p.grad.square().sum().item() for p in quantum.parameters()) ** 0.5
    )
    train_losses.append(loss.detach().item())
    optimizer.step()
    with torch.no_grad():
        test_prediction = model(x_test[:, None]).reshape(-1)
        test_losses.append((test_prediction - y_test).square().mean().item())

final_prediction = model(x_test[:, None]).detach().reshape(-1)
parameter_changes = {
    name: (p.detach() - initial_parameters[name]).norm().item()
    for name, p in model.named_parameters()
}
print("Test MSE:", test_losses[-1])
print("Parameter changes:", parameter_changes)
assert test_losses[-1] < 0.01
assert any(
    delta > 1e-5 for name, delta in parameter_changes.items() if name.startswith("0.")
)
assert any(
    delta > 1e-5 for name, delta in parameter_changes.items() if name.startswith("1.")
)


## 6. Read three pieces of evidence

The first panel checks predictions on held-out inputs. The second shows optimization progress.
The third shows gradients reaching both parts of the model. A good fit is useful, but no one panel alone establishes all three facts.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot(x_test, y_test, label="Target")
ax[0].scatter(x_test, initial_prediction, label="Before", s=15)
ax[0].scatter(x_test, final_prediction, label="After", s=15)
ax[0].set(xlabel="Held-out x", ylabel="Prediction")
ax[0].legend()
ax[1].semilogy(train_losses, label="Train (before update)")
ax[1].semilogy(test_losses, label="Test (after update)")
ax[1].set(xlabel="Step", ylabel="MSE")
ax[1].legend()
ax[2].plot(encoder_gradients, label="Classical encoder")
ax[2].plot(quantum_gradients, label="Quantum layer")
ax[2].set(xlabel="Step", ylabel="Gradient norm")
ax[2].legend()
plt.tight_layout()
plt.show()


## 7. Keep a result you can explain

Record the configuration and evidence, not just a plot. The source path helps you catch an unintended installation; do not publish private paths without reviewing them.

In [ ]:
from pathlib import Path

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "flagquantum").is_dir()
)
OUTPUTS = ROOT / "workshops/flagos2026/outputs"
OUTPUTS.mkdir(exist_ok=True)


In [ ]:
import json

report = {
    "experiment": "hybrid_cosine_fit",
    "source": "local_simulation",
    "flagquantum_version": fq.__version__,
    "torch_version": torch.__version__,
    "quantum_seed": 42,
    "steps": 150,
    "learning_rate": 0.04,
    "train_points": len(x_train),
    "test_points": len(x_test),
    "test_mse": test_losses[-1],
    "parameter_changes": parameter_changes,
    "model": "cos(w*x+b+offset)",
}
(OUTPUTS / "hybrid_model_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))


## Explain it to someone else

- Where is the gradient crossing from the quantum layer into the classical encoder?
- Why can different values of b and offset produce the same predictions? Only their sum appears in this model, so the individual values are not identifiable.
- What evidence would you need before claiming this model is better than a classical alternative?

**Try next:** freeze one component, restart from the same initial values, and compare results. Or test outside [-2,2] and label that experiment as extrapolation.
Continue to [lab 18](../simulation/18_one_problem_three_simulators.ipynb) to keep an objective fixed while changing its simulation representation.
Use [Field notes](../../FIELD_NOTES.md) to record your conclusion.